# সহজ ভাষায় Notebook Guide

এই notebook-এ project-এর **একই `src/` module** (app.py আর cli.py যেগুলো ব্যবহার করে) cell-by-cell চালিয়ে ReAct loop খুলে দেখা হয়েছে। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন।
2. প্রতিটা loop চালানোর আগে লিখে ফেলুন: কোন exit (`final`, `loop`, `max_steps`...) দিয়ে থামবে।
3. Section 6-8 real Ollama লাগে (`ollama serve` + `ollama pull llama3.1:8b`)। Ollama না থাকলে `llm_failed` দেখায়, crash করে না।

# Deterministic Function-Calling Travel Agent — Exploration

1. Deterministic tools — একই input, সবসময় একই output
2. Trust boundary — `execute()` কখনো exception ছোড়ে না
3. Approval gate — side effect-এর আগে human
4. Scripted fake LLM দিয়ে loop-এর প্রতিটা exit
5. Classic text ReAct parser
6. Real Ollama: native vs text mode, পাশাপাশি
7. llama3.1-এর template রহস্য — কেন `final_answer` tool আর nudge লাগে
8. Golden-task evaluation

In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging
import pandas as pd

from config import get_config
from src.agent.react_agent import NUDGE, parse_text_step, progress_note, run_agent
from src.evaluation.harness import GOLDEN_TASKS, evaluate
from src.llm.client import LLMUnavailable, chat
from src.tools.travel import TravelTools, tool_schemas

logging.getLogger("src.agent.react_agent").setLevel(logging.WARNING)   # notebook-এ log noise কম
cfg = get_config()
tools = TravelTools(cfg.data.flights_path)
print("model:", cfg.llm.model, "| airports:", tools.airports)

model: llama3.1:8b | airports: ['CGP', 'CXB', 'DAC', 'DXB', 'JSR', 'SPD']


## 1. Deterministic Tools

`search_flights` একটা pure function। Sort হয় `(price, departs, id)` total order-এ — BG101 আর US103 দুটোরই দাম 6500, কিন্তু ক্রম কখনো বদলায় না। Observation compact (৫টা field), যাতে context দ্রুত না ভরে।

In [2]:
r1 = tools.search_flights("DAC", "CXB", date="2026-11-05")
r2 = tools.search_flights("DAC", "CXB", date="2026-11-05")
print("identical:", r1 == r2)
pd.DataFrame(r1["flights"])

identical: True


,id,date,departs,price_bdt,seats_left
0,BG101,2026-11-05,08:00,6500,5
1,US103,2026-11-05,12:10,6500,2
2,BG102,2026-11-05,16:30,7200,0


## 2. Trust Boundary — `execute()`

Model-এর পাঠানো প্রতিটা call untrusted input। `execute()` আগে registry-তে নাম খোঁজে, তারপর Pydantic দিয়ে validate করে। ভুল হলে exception না, একটা structured observation (`ok`, `error`, `hint`) — model পরের turn-এ এটা পড়ে শুধরে নেয়।

In [3]:
bad = [("search_flight", {}),
       ("search_flights", {"from": "DAC", "to": "CXB"}),
       ("search_flights", {"origin": "DAC", "destination": "CXB", "date": "5 Nov"}),
       ("search_flights", {"origin": "dac", "destination": "cxb", "max_price": "6600"}),
       ("book_flight", "{broken json")]
for name, args in bad:
    obs = tools.execute(name, args)
    print(f"{name}({args})\n  -> {json.dumps(obs)[:170]}\n")

search_flight({})
  -> {"ok": false, "error": "unknown_tool", "tool": "search_flight", "valid_tools": ["book_flight", "search_flights"], "did_you_mean": ["search_flights"]}

search_flights({'from': 'DAC', 'to': 'CXB'})
  -> {"ok": false, "error": "invalid_arguments", "tool": "search_flights", "details": [{"field": "origin", "msg": "Field required"}, {"field": "destination", "msg": "Field req

search_flights({'origin': 'DAC', 'destination': 'CXB', 'date': '5 Nov'})
  -> {"ok": false, "error": "invalid_arguments", "tool": "search_flights", "details": [{"field": "date", "msg": "Value error, must be YYYY-MM-DD"}], "hint": "Correct the argum

search_flights({'origin': 'dac', 'destination': 'cxb', 'max_price': '6600'})
  -> {"ok": true, "count": 2, "flights": [{"id": "BG101", "date": "2026-11-05", "departs": "08:00", "price_bdt": 6500, "seats_left": 5}, {"id": "US103", "date": "2026-11-05", 

book_flight({broken json)
  -> {"ok": false, "error": "arguments_not_json", "hint": "Send arguments 

লক্ষ্য করুন চার নম্বর call: lowercase code আর `"6600"` string — Pydantic-এর lax mode এগুলো ঠিক করে নেয় (`DAC`, `6600`)। কিন্তু `{"from", "to"}` invented field — `extra="forbid"` থাকায় চুপচাপ বাদ না পড়ে error হয়।

## 3. Approval Gate

`book_flight` side-effecting। Booking-এর আগে `approve(details)` ডাকা হয় — এটা code-এ, prompt-এ না। নিচে "human" না বলে দিল।

In [4]:
tools.reset()
tools.approve = lambda d: (print("  human asked:", d["summary"]), False)[1]
print(tools.book_flight("BG101", "Rahim"))
print("bookings after denial:", tools.bookings)
tools.approve = lambda d: True
print(tools.book_flight("BG101", "Rahim"))
print(tools.book_flight("BG101", "Rahim")["error"], "<- idempotent: same passenger, same flight")
tools.reset()

  human asked: Rahim on BG101 DAC->CXB 2026-11-05 08:00, 6500 BDT
{'ok': False, 'error': 'approval_denied', 'summary': 'Rahim on BG101 DAC->CXB 2026-11-05 08:00, 6500 BDT', 'hint': 'Nothing was booked. Tell the user the booking needs their confirmation.'}
bookings after denial: []
{'ok': True, 'booking_ref': 'BK-BG101-RAHIM', 'flight_id': 'BG101', 'passenger': 'Rahim', 'price_bdt': 6500, 'status': 'CONFIRMED'}
duplicate_booking <- idempotent: same passenger, same flight


## 4. Loop-এর প্রতিটা Exit — scripted fake LLM

Fake LLM আগে থেকে লেখা reply ফেরত দেয়, তাই প্রতিটা exit হুবহু ঘটিয়ে দেখা যায়। Paper-এর error analysis-এ ReAct-এর ৪৭% failure ছিল reasoning loop — এই guard-গুলো সেখান থেকেই।

In [5]:
def call(name, **a): return {"name": name, "arguments": a}
def say(text="", *calls): return {"content": text, "tool_calls": list(calls)}

def scripted(*replies):
    it = iter(replies)
    return lambda messages, **kw: next(it)

search = call("search_flights", origin="DAC", destination="CXB", date="2026-11-05")
scenarios = {
    "happy path": scripted(say("", search), say("", call("book_flight", flight_id="BG101", passenger_name="Rahim")),
                           say("", call("final_answer", answer="Booked BK-BG101-RAHIM"))),
    "same call x3": scripted(say("", search), say("", search), say("", search)),
    "invalid x3": scripted(*[say("", call("fly", n=i)) for i in range(3)]),
    "never stops": scripted(*[say("", call("search_flights", origin="DAC", destination="CXB", max_price=9000 + i)) for i in range(9)]),
}
rows = []
for name, llm in scenarios.items():
    tools.reset()
    res = run_agent("Book cheapest DAC-CXB 2026-11-05 for Rahim", tools, llm, max_steps=8)
    rows.append({"scenario": name, "status": res.status, "llm_calls": res.llm_calls, "answer": res.answer[:90]})
pd.DataFrame(rows)

,scenario,status,llm_calls,answer
0,happy path,final,3,Booked BK-BG101-RAHIM
1,same call x3,loop,3,I couldn't finish (kept repeating the same act...
2,invalid x3,error_budget,3,I couldn't finish (made too many invalid tool ...
3,never stops,max_steps,8,I couldn't finish (ran out of steps). Nothing ...


`loop` আর `max_steps`-এ উত্তরটা **deterministic fallback** — আরেকটা LLM call না, শেষ সফল observation থেকে বানানো ("Nothing was booked. Flights I found: ...")। থামার পরও user সৎ, কাজের উত্তর পায়।

## 5. Classic Text ReAct Parser

২০২২-এর ReAct format: `Thought / Action / Action Input`। Parser markdown fence আর বাড়তি লেখা সহ্য করে; format না মানলে `ValueError` → loop সেটাকে `parse_error` observation বানায়।

In [6]:
samples = ['Thought: need flights\nAction: search_flights\nAction Input: {"origin": "DAC", "destination": "CGP"}',
           'Thought: x\nAction: search_flights\nAction Input: ```json\n{"origin": "DAC"}\n```\nI hope this works!',
           'Thought: all done\nFinal Answer: BG201 departs at 09:15',
           'I will search for flights now.']
for s in samples:
    try:
        print(parse_text_step(s))
    except ValueError as e:
        print("ValueError:", e)

{'thought': 'need flights', 'action': 'search_flights', 'input': '{"origin": "DAC", "destination": "CGP"}'}
{'thought': 'x', 'action': 'search_flights', 'input': '{"origin": "DAC"}'}
{'thought': 'all done', 'final': 'BG201 departs at 09:15'}
ValueError: reply has neither 'Action:'/'Action Input:' nor 'Final Answer:'


## 6. Real Ollama — Native vs Text Mode

একই task দুই mode-এ। প্রতিটা step-এ `prompt_tokens` দেখুন — history বাড়ার সাথে context কীভাবে বাড়ে।

In [7]:
def live(task, mode="native", model=None, **kw):
    tools.reset()
    try:
        res = run_agent(task, tools, chat, mode=mode, model=model, **kw)
    except LLMUnavailable as e:
        print("Ollama unavailable:", e); return None
    print(f"[{mode}] {task}\n  -> {res.status} ({res.llm_calls} LLM calls): {res.answer}")
    for s in res.steps:
        print(f"     {s.n}. {s.tool}({json.dumps(s.args, default=str)[:80]}) -> "
              f"{json.dumps(s.observation)[:80]}  [prompt_tokens={s.prompt_tokens}]")
    print("  bookings:", [b["booking_ref"] for b in tools.bookings], "\n")

TASK = "Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim"
live(TASK, "native")
live(TASK, "text")

[native] Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim
  -> final (4 LLM calls): Flight booked successfully. Booking reference: BK-BG101-RAHIM
     1. search_flights({"max_price": null, "origin": "DAC", "date": "2026-11-05", "destination": "CXB"}) -> {"ok": true, "count": 3, "flights": [{"id": "BG101", "date": "2026-11-05", "depa  [prompt_tokens=722]
     2. book_flight({"flight_id": "BG101", "passenger_name": "Rahim"}) -> {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger":  [prompt_tokens=1049]
     3. book_flight({"flight_id": "BG101", "passenger_name": "Rahim"}) -> {"ok": false, "error": "repeated_call", "previous_result": {"ok": true, "booking  [prompt_tokens=1294]
  bookings: ['BK-BG101-RAHIM'] 



[text] Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim
  -> final (4 LLM calls): Your flight from DAC to CXB on 2026-11-05 has been booked successfully. Your booking reference is BK-BG101-RAHIM.
     1. search_flights("{\"origin\": \"DAC\", \"destination\": \"CXB\", \"date\": \"2026-11-05\"}") -> {"ok": true, "count": 3, "flights": [{"id": "BG101", "date": "2026-11-05", "depa  [prompt_tokens=433]
     2. search_flights("{\"origin\": \"DAC\", \"destination\": \"CXB\", \"date\": \"2026-11-05\", \"max) -> {"ok": true, "count": 2, "flights": [{"id": "BG101", "date": "2026-11-05", "depa  [prompt_tokens=656]
     3. book_flight("{\"flight_id\": \"BG101\", \"passenger_name\": \"Rahim\"}") -> {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger":  [prompt_tokens=847]
  bookings: ['BK-BG101-RAHIM'] 



## 7. llama3.1 Template রহস্য — কেন `final_answer` tool আর nudge

`ollama show llama3.1:8b --template` দেখলে দুটো জিনিস বোঝা যায়:

1. Tool list শুধু **শেষ user message**-এর পাশে দেখানো হয়। Tool result আসার পর শেষ message আর user না — model আর tool দেখতে পায় না, তাই "I will book BG101" লেখে কিন্তু `book_flight` ডাকে না।
2. Tool দেখানো হলে template বলে "respond with a JSON for a function call" — model তখন plain text-এ উত্তর দিতে পারে না।

সমাধান: একটা `final_answer` tool (smolagents-এর `ToolCallingAgent`-এর মতো), আর প্রতিটা observation-এর পরে একটা ছোট user nudge যেটা মূল request আর "already done" তালিকা মনে করিয়ে দেয়। নিচে nudge বন্ধ করে একই task চালাই।

In [8]:
live(TASK, "native", nudge_after_tools=False)
print("The nudge the model sees after a search step looks like this:\n")
fake_steps = run_agent(TASK, tools, scripted(say("", search), say("done"))).steps
print(NUDGE.format(request=TASK, done=progress_note(fake_steps)))

[native] Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim
  -> final (2 LLM calls): Would you like to book flight BG101?
     1. search_flights({"date": "2026-11-05", "destination": "CXB", "max_price": null, "origin": "DAC"}) -> {"ok": true, "count": 3, "flights": [{"id": "BG101", "date": "2026-11-05", "depa  [prompt_tokens=722]
  bookings: [] 

The nudge the model sees after a search step looks like this:

My request: Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim
Already done (do not repeat these):
1. search_flights({'origin': 'DAC', 'destination': 'CXB', 'date': '2026-11-05'}) -> BG101 6500 BDT (5 seats), US103 6500 BDT (2 seats), BG102 7200 BDT (0 seats)
If my request still needs an action that is not done yet, call that tool now. Otherwise call final_answer with your reply to me.


### Model Reference Table (BFCL v4, 2026-04-12)

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| `llama3.1:8b` (default) | Course standard | Native tool calling | Multi-turn ~11% — workaround ছাড়া chain করতে পারে না |
| `qwen3:8b` | ভালো local agent | Multi-turn ~42% | Course standard না |
| `llama3.2:3b` | 8GB RAM | ছোট, দ্রুত | Multi-turn ~4% |

In [9]:
live(TASK, "native", model="llama3.2:3b")

[native] Book the cheapest DAC to CXB flight on 2026-11-05 for Rahim
  -> loop (4 LLM calls): I stopped early (kept repeating the same action), but this booking is confirmed: BK-BG101-RAHIM (BG101, Rahim).
     1. search_flights({"date": "2026-11-05", "origin": "DAC", "destination": "CXB", "max_price": null}) -> {"ok": true, "count": 3, "flights": [{"id": "BG101", "date": "2026-11-05", "depa  [prompt_tokens=719]
     2. search_flights({"date": "2026-11-05", "origin": "DAC", "destination": "CXB", "max_price": null}) -> {"ok": false, "error": "repeated_call", "previous_result": {"ok": true, "count":  [prompt_tokens=1046]
     3. book_flight({"flight_id": "BG101", "passenger_name": "Rahim"}) -> {"ok": true, "booking_ref": "BK-BG101-RAHIM", "flight_id": "BG101", "passenger":  [prompt_tokens=1468]
     4. search_flights({"date": "2026-11-05", "origin": "DAC", "destination": "CXB", "max_price": null}) -> {"ok": false, "error": "repeated_call"}  [prompt_tokens=1770]
  bookings: ['BK-BG101-RAH

## 8. Golden-Task Evaluation

শুধু final answer না — তিন স্তরে grade: (১) run শেষে booking state, (২) আবশ্যক/নিষিদ্ধ tool call (trajectory + policy), (৩) answer-এ দরকারি শব্দ। Approval gate-এ একটা **simulated user** বসে — সে শুধু তার নিজের request-এ লেখা নামের জন্য booking approve করে, তাই বানানো "John Doe" আটকে যায়।

In [10]:
summaries = {}
for mode in ("native", "text"):
    rep = evaluate(chat, TravelTools(cfg.data.flights_path), k=1, mode=mode)
    summaries[mode] = rep.summary()
    print(f"--- {mode} ---")
    for t in rep.trials:
        print(f"  {'PASS' if t.ok else 'FAIL'}  {t.task:<18} {t.status:<12} {t.reason}")
pd.DataFrame(summaries).T[["success_rate", "pass^k", "invalid_call_rate", "avg_llm_calls", "avg_seconds"]]

--- native ---
  PASS  book_cheapest_tie  final        
  PASS  book_single_seat   final        
  PASS  search_only        final        
  PASS  cheapest_return    final        
  FAIL  sold_out           loop         stopped with loop
  PASS  unknown_city       final        
  FAIL  missing_name       final        policy: should not call ['book_flight']
  PASS  over_budget        final        


--- text ---
  PASS  book_cheapest_tie  final        
  PASS  book_single_seat   final        
  FAIL  search_only        final        answer missing ['BG201']
  FAIL  cheapest_return    final        policy: should not call ['book_flight']
  FAIL  sold_out           final        bookings [('BG201', 'Karim')] != expected []
  PASS  unknown_city       final        
  FAIL  missing_name       final        policy: should not call ['book_flight']
  PASS  over_budget        final        


,success_rate,pass^k,invalid_call_rate,avg_llm_calls,avg_seconds
native,0.75,0.75,0.143,2.62,24.61
text,0.5,0.5,0.481,4.38,51.97


**কী শিখলাম:**

* Native mode text mode-এর চেয়ে বেশি নির্ভরযোগ্য — argument dict হিসেবে আসে, model নকল Observation লিখতে পারে না।
* ছোট model-এ সাফল্যের বড় অংশ আসে model-এর চারপাশের code থেকে: validation, repeat guard, approval gate, template-aware nudge।
* `temperature=0` + একক local Ollama-তে একই task বারবার চালালে প্রায় হুবহু একই trajectory আসে — তাই এখানে pass^k ≈ pass^1। আসল reliability মাপতে prompt-এর ভিন্ন phrasing দিয়ে task বাড়াতে হয়।

## 🧠 Brain Teasers

1. `max_repeat_calls=1` করলে "same call x3" scenario কোন step-এ থামে? কেন?
2. Golden set-এ একটা নতুন task যোগ করুন: "Book BS401 for Rahim and Karim"। কোন grade check দরকার, আর agent কোথায় ভুল করে?
3. Text mode-এ `stop` sequence সরিয়ে দিলে (client-এ `stop=None`) কী ঘটে? Model কি নিজের Observation বানায়?